# Si/graphite degradation model: reproducing the paper results

One shared parameter set, four test conditions:

| Condition | Cell | Temperature | SoC window |
|---|---|---|---|
| `25deg_full_soc` | CELL064 | 25 °C | full (2.5–4.195 V) |
| `45deg_full_soc` | CELL017 | 45 °C | full |
| `25deg_15-95_soc` | CELL009 | 25 °C | 15–95% (3.14–4.15 V) |
| `25deg_20-80_soc` | CELL026 | 25 °C | 20–80% (3.969 V top, fixed 1.5 Ah discharge) |

The model is a modified PyBaMM DFN with these additions:
- a composite Si/graphite negative electrode;
- SEI, including SEI on cracks;
- Si particle cracking by two routes: a Paris law on the diffusion stress, plus contraction strain fatigue;
- stress-driven LAM;
- porosity-gated isolation LAM;
- pore buffering of the Si swelling (part of the swelling is taken up by the pore volume).

The model equations are in `docs/MODEL_EQUATIONS.md` and the code changes in `docs/CHANGES.md`.

**What the notebook does:**
1. sets up the modified PyBaMM and the `sigr/` package;
2. shows the model options, the shared parameter set and the four conditions;
3. **simulates all four conditions**;
4. scores the results against the data;
5. plots the figures.

**Inputs:**
- this repository: the `sigr/` package next to this notebook;
- the experimental data (Zenodo DOI: *to be added*), unpacked to `data/`.

**Runtime:** on one CPU core, the 25 °C full-window condition takes about 20 min and the other three about 35–40 min each. Each needs about 1 GB of RAM. The conditions run in parallel, one process per core:
- a 4-core machine needs about 40 min in total;
- free Google Colab (2 cores) needs about 1–1.5 h.

To skip the simulations, set `RUN_SIMULATIONS = False` in section 2. The notebook then loads the paper's precomputed results, shipped in `results/`, or your own earlier results in `outputs/` if they exist.

## 1. Install the modified PyBaMM

Use a fresh environment: `conda env create -f environment.yml`. The modified PyBaMM installs as `pybamm`, so it replaces any official PyBaMM in that environment. To install by hand, uncomment one of the two options below, then restart the kernel.

In [ ]:
# From GitHub (needs git), pinned to the paper's tag:
# %pip install "git+https://github.com/Dharshannan/PyBaMM.git@si-gr-degradation-v1.0#subdirectory=packages/pybamm"

# From the source archive in the Zenodo record (no .git, so the version must be given):
# import os; os.environ["SETUPTOOLS_SCM_PRETEND_VERSION"] = "26.7.1+si-gr-degradation-v1.0"
# %pip install ./pybamm-si-gr-degradation-v1.0/packages/pybamm

### Google Colab

On Colab, run the next cell first. It:
1. clones this repository at the paper tag (`REF`), for the `sigr/` package next to the notebook;
2. installs the modified PyBaMM from that clone;
3. asks you to upload the data zip (containing `data/`) and unpacks it next to the notebook.

Re-running the cell is safe: it skips the install, and updates the clone to `REF`. If Colab itself asks to restart (after upgrading a preinstalled package), restart and run this cell again before the rest. Keep the browser tab open while the simulations run, because Colab disconnects idle sessions.

In [ ]:
import os, subprocess, sys

REF = "si-gr-degradation-v1.0"  # tag (or branch) of this repository to use
if "google.colab" in sys.modules:
    repo, url = "/content/PyBaMM", "https://github.com/Dharshannan/PyBaMM.git"
    if not os.path.isdir(repo):  # full history (blobs on demand), so PyBaMM's version tags resolve
        subprocess.run(["git", "clone", "--filter=blob:none", url, repo], check=True)
    # Always check out REF, so the notebook, sigr/ and PyBaMM come from the same commit
    subprocess.run(["git", "-C", repo, "fetch", "--tags", "--force", "origin", REF], check=True)
    subprocess.run(["git", "-C", repo, "checkout", "--force", "FETCH_HEAD"], check=True)
    print("checked out", REF, "at",
          subprocess.run(["git", "-C", repo, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
    # Editable install (dependencies + version file), then put the checked-out source first on the
    # path: this session imports it straight away, and so do the simulation worker processes.
    src = f"{repo}/packages/pybamm/src"
    check = f"import pybamm, os; assert os.path.realpath(pybamm.__file__).startswith(os.path.realpath('{repo}'))"
    if subprocess.run([sys.executable, "-c", check], capture_output=True).returncode:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo}/packages/pybamm"], check=True)
    if src not in sys.path:
        sys.path.insert(0, src)
    os.chdir(f"{repo}/paper")
    if not os.path.isdir("data"):
        from google.colab import files
        for name in files.upload():  # the data zip, containing data/
            subprocess.run(["unzip", "-q", "-o", name, "-d", "."], check=True)
    print("working directory:", os.getcwd(), "| data:", os.path.isdir("data"))


In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt
import pybamm

sys.path.insert(0, os.path.abspath("."))  # the sigr package next to this notebook
import sigr

print("pybamm", pybamm.__version__, "from", os.path.dirname(pybamm.__file__))
assert "porosity isolation gate" in pybamm.BatteryModelOptions({}).possible_options, \
    "this is not the modified PyBaMM -- see section 1"

## 2. Settings

**C-rate and EFC basis.** The paper runs use the model's capacity, 2.5947 Ah, for both:
- C/3 = 0.865 A;
- EFC = throughput / (2 × 2.5947 Ah).

The tests define both on the cells' 2.5 Ah nominal capacity (C/3 = 0.833 A). Set both to 2.5 to simulate the test-protocol currents; that changes the mean SoH gaps by up to 1.1 pp.

In [ ]:
DATA_DIR = os.path.abspath("data")        # the data record
OUT_DIR = os.path.abspath("outputs")      # new model results, logs and figures
PAPER_RESULTS_DIR = os.path.abspath("results")  # the paper's precomputed results (in the repository)
FIG_DIR = os.path.join(OUT_DIR, "figures")
RUN_SIMULATIONS = True                    # False: load results (OUT_DIR if there, else the paper's results/)
MAX_WORKERS = None                        # parallel runs (default: one per CPU core, up to 4)
RATE_BASIS_AH = 2.5947                    # C-rate basis [A.h] (paper: 2.5947; test protocol: 2.5)
EFC_BASIS_AH = 2.5947                     # EFC basis [A.h]     (paper: 2.5947; data: 2.5)
KEYS = list(sigr.CONDITIONS)

## 3. Model options, parameters and conditions

In [ ]:
pd.Series({k: str(v) for k, v in sigr.MODEL_OPTIONS.items()}, name="option").to_frame()

In [ ]:
pd.Series(sigr.V6, name="value").to_frame()  # shared degradation parameters (v6), at 25 degC

In [ ]:
pd.DataFrame([dict(key=c.key, cell=c.cell, T_degC=c.temperature_K - 273.15, window=c.window,
                   F0=c.F0, width=c.width, ageing_cycle=" | ".join(sigr.conditions.ageing_cycle(c, RATE_BASIS_AH)))
              for c in sigr.CONDITIONS.values()]).set_index("key")

## 4. Simulate the four conditions

Each condition runs in this order:
1. formation;
2. C/3 ageing in its own SoC window;
3. a C/20 RPT at each real RPT's EFC (aligned RPTs).

Progress is printed every 5 min, and each condition's full log is in `outputs/<condition>_log.txt`. The results are saved to `outputs/<condition>_results.npz` and `<condition>_rpt_curves.csv`, and everything below is computed from them.

With `RUN_SIMULATIONS = False`, each condition is loaded from `outputs/` if you have simulated it, and otherwise from the paper's precomputed `results/`.

In [ ]:
if RUN_SIMULATIONS:
    sigr.run_all(DATA_DIR, OUT_DIR, KEYS, RATE_BASIS_AH, EFC_BASIS_AH, max_workers=MAX_WORKERS)

def results_dir(key):
    """outputs/ if this condition has been simulated there, otherwise the paper's results/."""
    return OUT_DIR if os.path.exists(os.path.join(OUT_DIR, f"{key}_results.npz")) else PAPER_RESULTS_DIR

results = {}
for k in KEYS:
    results[k] = sigr.analysis.load(k, results_dir(k))
    print(f"{k:<17} loaded from {results_dir(k)}")

## 5. Scores

Score definitions (the full tables are in section 6):

| Score | Definition |
|---|---|
| SoH gap | model C/20 RPT SoH, interpolated at each real C/20 RPT's EFC, minus the real value; mean of \|gap\| |
| Voltage RMSE | C/20 discharge V(Q) against the nearest model RPT |
| Knee | steepest descent of the model's C/3 SoH, and its SoH-91% crossing |

In [ ]:
scores = pd.DataFrame([sigr.analysis.summary(*results[k], sigr.CONDITIONS[k], DATA_DIR) for k in KEYS]).set_index("condition")
scores.round(3)

## 6. Per-condition tables

In [ ]:
for k in KEYS:
    cond, (res, curves) = sigr.CONDITIONS[k], results[k]
    print(f"\n=== {cond.label} ===")
    print("SoH at real C/20 RPTs [%]:\n", sigr.analysis.rpt_gaps(res, cond, DATA_DIR).round(2).to_string(index=False))
    print("LLI and LAM at real RPTs [%]:\n", sigr.analysis.lli_lam_table(res, cond, DATA_DIR).round(1).to_string(index=False))
    print("Si cracks at model RPTs:\n", sigr.analysis.crack_table(res).round(3).to_string(index=False))

## 7. Figures

The figures are saved to `outputs/figures/<condition>_<figure>.png`:
- **overall summary:** SoH, LAM and LLI, C/20 RPT voltage;
- **RPT voltage and dV/dSoC:** each real C/20 RPT against its nearest model RPT, against depth of discharge;
- **crack diagnostics:** Si crack length and roughness, peak stress, negative-electrode porosity (minimum of each ageing cycle, i.e. without the within-cycle swelling oscillation), Si SEI, and the growth from each cracking route.

In [ ]:
from IPython.display import Image, display

for k in KEYS:
    cond, (res, curves) = sigr.CONDITIONS[k], results[k]
    paths = sigr.plots.save_all(res, curves, cond, DATA_DIR, FIG_DIR)
    for name, path in paths.items():
        display(Image(filename=path))